# 04 — Instrumentos y variantes de liquidación
**Spec:** docs/specs/04_instrumentos_mapeo.md · **Requisitos:** R-22 · **Escribe:** `iol_ops.instrument_mapping`, y reaplica `iol_silver.trades`

Una fila por símbolo: tipo de activo, variante de liquidación, subyacente y proveedor. El sufijo D/C no alcanza: `YPFD` está en pesos (ADR-004).
Lo que las reglas no resuelven queda en `NEEDS_REVIEW` para la IA (notebook 09). El proveedor y la escala quedan `UNVERIFIED` hasta el notebook 05.

In [0]:
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("demo_rerun", "true", ["true", "false"])

import json
import os
import sys
import uuid

import requests

sys.path.insert(0, os.path.abspath("../src"))
from iol_pipeline import instruments, io, silver

RUN_ID = dbutils.widgets.get("pipeline_run_id") or str(uuid.uuid4())
CATALOG = dbutils.widgets.get("catalog")
print("pipeline_run_id:", RUN_ID)

io.ensure_schemas(spark, catalog=CATALOG)
bronze_table = io.table_name("bronze", "trades_raw", catalog=CATALOG)
silver_table = io.table_name("silver", "trades", catalog=CATALOG)
mapping_table = io.table_name("ops", "instrument_mapping", catalog=CATALOG)
enrichment_table = io.table_name("ops", "instrument_enrichment", catalog=CATALOG)  # la escribe el notebook 09
quote_series_table = io.table_name("ops", "quote_series", catalog=CATALOG)  # la escribe el notebook 05
baselines_table = io.table_name("ops", "quality_baselines", catalog=CATALOG)
events_table = io.table_name("ops", "quality_events", catalog=CATALOG)

pipeline_run_id: b843c858-3f56-4fd4-bfab-a65057b32d1e


## Corrida
Después del mapeo se reaplica Silver: `DQ030` y `DQ033` lo necesitan, y Silver corre antes. Con `demo_rerun = true`, la segunda corrida del mapeo tiene que dar 0 insertadas y 0 actualizadas.

In [0]:
first = instruments.run_instruments(spark, silver_table, mapping_table, enrichment_table=enrichment_table, quote_series_table=quote_series_table)
print("1ra corrida:", first)
second = None
if dbutils.widgets.get("demo_rerun") == "true":
    second = instruments.run_instruments(spark, silver_table, mapping_table, enrichment_table=enrichment_table, quote_series_table=quote_series_table)
    print("2da corrida:", second)
reapplied = silver.run_silver(spark, bronze_table, silver_table, baselines_table, events_table, RUN_ID, mapping_table)
print("Silver reaplicado:", reapplied)

1ra corrida: {'rows_inserted': 0, 'rows_updated': 0, 'symbols': 1445}
2da corrida: {'rows_inserted': 0, 'rows_updated': 0, 'symbols': 1445}
Silver reaplicado: {'bronze_rows': 100000, 'silver_rows': 100000, 'silver_inserted': 0, 'silver_updated': 0, 'baselines_updated': 0, 'baselines_inserted': 0, 'events_inserted': 0, 'events_updated': 0}


## 1. Símbolos por tipo, moneda y origen de la clasificación

In [0]:
mapping = spark.table(mapping_table)
trades = spark.table(silver_table)
display(instruments.type_summary(mapping))
print("Símbolos en el mapeo:", mapping.count(), "· en Silver:", trades.select("symbol").distinct().where("symbol IS NOT NULL").count())

asset_type,currency_code,asset_type_source,symbols
ACCION_LOCAL,ARS,AI,1
ACCION_LOCAL,ARS,RULE,65
ACCION_LOCAL,USD,RULE,22
BONO_SOBERANO,ARS,MANUAL,4
BONO_SOBERANO,ARS,RULE,53
BONO_SOBERANO,USD,RULE,41
BONO_SUBSOBERANO,ARS,MANUAL,1
BONO_SUBSOBERANO,ARS,RULE,10
BONO_SUBSOBERANO,USD,AI,1
BONO_SUBSOBERANO,USD,RULE,7


Símbolos en el mapeo: 1445 · en Silver: 1445


## 2. Variantes de liquidación
`implied_fx` = mediana en ARS / mediana de la variante: el MEP o CCL implícito. Si es coherente entre instrumentos, las variantes están bien identificadas.

In [0]:
variants = instruments.variants_with_implied_fx(mapping, trades)
display(variants)
print("Variantes con cambio implícito fuera de rango:")
display(instruments.variants_fx_out_of_range(variants))

underlying_key,symbol,currency_code,settlement_variant,asset_type,median_price,implied_fx,ops
AAL,AAL,ARS,ARS,CEDEAR,10615.79035,null,238
AAL,AALC,USD,USD_CCL,CEDEAR,6.7013,1584.14,1
AAL,AALD,USD,USD_MEP,CEDEAR,7.38645,1437.2,30
AAPL,AAPL,ARS,ARS,CEDEAR,19625.8112,null,1110
AAPL,AAPLC,USD,USD_CCL,CEDEAR,13.1884,1488.11,1
AAPL,AAPLD,USD,USD_MEP,CEDEAR,13.4356,1460.73,152
ABBV,ABBV,ARS,ARS,CEDEAR,33585.6358,null,33
ABBV,ABBVD,USD,USD_MEP,CEDEAR,23.2539,1444.3,3
ABEV,ABEV,ARS,ARS,CEDEAR,12691.329249999999,null,54
ABEV,ABEVD,USD,USD_MEP,CEDEAR,7.85365,1615.98,6


Variantes con cambio implícito fuera de rango:


underlying_key,symbol,currency_code,settlement_variant,asset_type,median_price,implied_fx,ops
TEAM,TEAMD,USD,USD_MEP,CEDEAR,1.7139,2208.46,3


## 3. Falsos positivos del sufijo: terminan en D o C y están en pesos

In [0]:
display(instruments.suffix_false_positives(mapping))

symbol,description_canonical,asset_type,settlement_variant
AMD,"Cedear Advanced Micro Devices,",CEDEAR,ARS
BA.C,Cedear Bank Of America Corpora,CEDEAR,ARS
BA37D,Bono Pcia. Bs. As REGS New U$S 2037 A,BONO_SUBSOBERANO,ARS
BB37D,Bono Pcia. Bs. As. REGS New 2037 B,BONO_SUBSOBERANO,ARS
BBD,Cedear Banco Bradesco S.A.,CEDEAR,ARS
BC37D,Bono Pcia Bs As Regs New U$S 2037 C,BONO_SUBSOBERANO,ARS
C,Cedear Citigroup,CEDEAR,ARS
CARC,Carboclor S.A.,ACCION_LOCAL,ARS
DD,Cedear E.I.Du Pont The Nemours,CEDEAR,ARS
ERIC,Cedear L.M. Ericsson Telephone,CEDEAR,ARS


## 4. `NEEDS_REVIEW` para la IA

In [0]:
pending = instruments.needs_review(mapping)
print("Símbolos en NEEDS_REVIEW:", pending.count())
display(pending)

Símbolos en NEEDS_REVIEW: 0


symbol,currency_code,description_canonical,mapping_status


## 5. Validación independiente contra data912
Compara la clasificación con las listas por tipo de data912, que son de hoy. La de acciones no cuenta: es la misma que usa la regla.

In [0]:
display(instruments.agreement_with_provider_lists(spark, mapping, requests.get))

list_name,independent,symbols_in_dataset,agree,needs_review,disagree,disagree_examples,error
arg_cedears,true,590,590,0,0,,null
arg_bonds,true,97,97,0,0,,null
arg_notes,true,2,2,0,0,,null
arg_corp,true,241,241,0,0,,null
arg_stocks,false,87,87,0,0,,null


## 6. Reconciliación de Silver después de reaplicar

In [0]:
check = silver.reconciliation(spark.table(bronze_table), trades, spark.table(events_table))
for key, value in check.items():
    print(f"{key:30} {value:,}")
reconciled = check["bronze_rows"] == check["silver_rows"] and not any(v for k, v in check.items() if k not in ("bronze_rows", "silver_rows"))
print("Reconciliación:", "OK" if reconciled else "CON DIFERENCIAS")

bronze_rows                    100,000
silver_rows                    100,000
bronze_not_in_silver           0
silver_not_in_bronze           0
not_ok_without_flag            0
severe_flags_without_event     0
active_events_without_flag     0
Reconciliación: OK


In [0]:
idempotent = second is None or second["rows_inserted"] == second["rows_updated"] == 0
complete = first["symbols"] == trades.select("symbol").distinct().where("symbol IS NOT NULL").count()
dbutils.notebook.exit(json.dumps({"status": "SUCCESS" if idempotent and complete and reconciled else "FAILED", "pipeline_run_id": RUN_ID, "rows_out": first["symbols"]}))